# Full BFCL V4 Evaluation: V1-V5 across all buckets

Runs `python -m src.evaluation.bfcl` per (variant, bucket) and writes canonical `results/<variant>/scores.json` with merge across runs (so a session disconnect mid-eval doesn't drop earlier buckets).

**Buckets**: `non_live`, `live`, `hallucination`, `multi_turn`, `agentic` (Agentic skipped by default; requires `SERPER_API_KEY`).

**Adapters**: pulled from HF Hub (`arnursb/info290-{v2-sft,v3-sft-masked,v4-grpo,v5-grpo-masked}`). Downloaded to `adapters/<variant>/` on first run, cached after.

**Runtime**: ~30-90 min per (variant, bucket) on T4. Pro/A100 about 2x faster. Budget ~15 GPU-hours for the full matrix V2-V5 across non_live + live + hallucination + multi_turn.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
gh_token = userdata.get('GITHUB_TOKEN')
!git clone https://{gh_token}@github.com/arnursabet/info290-function-calling-slm.git /content/repo
%cd /content/repo

!pip install -q -r requirements.txt pyyaml vllm==0.15.1

In [ ]:
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
try:
    os.environ['SERPER_API_KEY'] = userdata.get('SERPER_API_KEY')
    print('Serper API set; agentic eval will run.')
except Exception:
    print('No SERPER_API_KEY; agentic bucket will be skipped.')

from huggingface_hub import HfApi
print(f'HF auth: {HfApi().whoami()["name"]}')

!bfcl --version

In [ ]:
import json, subprocess
from pathlib import Path
from huggingface_hub import snapshot_download

BUCKETS_DEFAULT = ['hallucination', 'live', 'non_live', 'multi_turn']
BUCKETS_WITH_AGENTIC = BUCKETS_DEFAULT + ['agentic']

def download_adapter(repo_id: str, variant: str) -> str:
    local = Path('adapters') / variant
    if local.exists() and (local / 'adapter_config.json').exists():
        print(f'{variant}: adapter already cached at {local}')
        return str(local)
    snapshot_download(repo_id=repo_id, local_dir=str(local), token=os.environ['HF_TOKEN'])
    print(f'{variant}: downloaded {repo_id} to {local}')
    return str(local)

def existing_score(variant: str, bucket: str):
    p = Path(f'results/{variant}/scores.json')
    if not p.exists():
        return None
    return (json.load(open(p)).get('top_level') or {}).get(bucket)

def run_bucket(variant: str, bucket: str, adapter_path: str | None = None):
    prior = existing_score(variant, bucket)
    if prior is not None:
        print(f'[{variant}/{bucket}] skip (already {prior:.4f})')
        return prior
    print(f'[{variant}/{bucket}] running...')
    cmd = ['python', '-m', 'src.evaluation.bfcl', '--variant', variant, '--test-category', bucket]
    if adapter_path:
        cmd += ['--adapter', adapter_path]
    subprocess.run(cmd, check=True)
    return existing_score(variant, bucket)

def run_variant(variant: str, repo_id: str | None = None, buckets: list[str] = BUCKETS_DEFAULT):
    adapter = download_adapter(repo_id, variant) if repo_id else None
    results = {}
    for b in buckets:
        results[b] = run_bucket(variant, b, adapter)
    print(f'\n{variant} summary: {results}')
    return results

## V1: Qwen2.5-3B-Instruct zero-shot

Non-Live already evaluated (committed to `results/v1/scores.json`). This cell extends V1 with the remaining buckets via merge.

In [ ]:
v1_results = run_variant('v1', repo_id=None, buckets=['hallucination', 'live', 'multi_turn'])

## V2: SFT, unmasked

In [ ]:
v2_results = run_variant('v2', repo_id='arnursb/info290-v2-sft', buckets=BUCKETS_DEFAULT)

## V3: SFT, masked

In [ ]:
v3_results = run_variant('v3', repo_id='arnursb/info290-v3-sft-masked', buckets=BUCKETS_DEFAULT)

## V4: GRPO cold-start, unmasked

In [ ]:
v4_results = run_variant('v4', repo_id='arnursb/info290-v4-grpo', buckets=BUCKETS_DEFAULT)

## V5: GRPO cold-start, masked

In [ ]:
v5_results = run_variant('v5', repo_id='arnursb/info290-v5-grpo-masked', buckets=BUCKETS_DEFAULT)

## Comparison table

Reads each `results/<variant>/scores.json` and writes `results/comparison.md` and `results/comparison.json` for the report.

In [ ]:
import json
from pathlib import Path

VARIANTS = ['v1', 'v2', 'v3', 'v4', 'v5']
BUCKETS = ['non_live', 'live', 'hallucination', 'multi_turn', 'agentic']

rows = []
for v in VARIANTS:
    p = Path(f'results/{v}/scores.json')
    if not p.exists():
        rows.append({'variant': v, 'overall': None, **{b: None for b in BUCKETS}})
        continue
    s = json.load(open(p))
    rows.append({'variant': v, 'overall': s.get('overall'), **{b: (s.get('top_level') or {}).get(b) for b in BUCKETS}})

Path('results').mkdir(exist_ok=True)
json.dump(rows, open('results/comparison.json', 'w'), indent=2)

def fmt(x): return f'{x*100:.2f}%' if x is not None else '-'
header = '| Variant | ' + ' | '.join(b.replace('_', ' ').title() for b in BUCKETS) + ' | Overall |'
sep = '|' + '|'.join(['---'] * (len(BUCKETS) + 2)) + '|'
lines = [header, sep]
for r in rows:
    lines.append('| ' + r['variant'].upper() + ' | ' + ' | '.join(fmt(r[b]) for b in BUCKETS) + ' | ' + fmt(r['overall']) + ' |')
Path('results/comparison.md').write_text('\n'.join(lines) + '\n')
print('\n'.join(lines))

## Persist results to GitHub

After Drive disconnect or session crash, the canonical scores live in git. Run after each session that produces new buckets.

In [ ]:
!git config user.email 'arnur+colab@berkeley.edu'
!git config user.name 'Arnur (Colab)'
!git add results/
!git diff --cached --stat
!git commit -m 'eval: BFCL V4 results from Colab session' || echo 'nothing to commit'
!git push origin HEAD || echo 'push failed; commit anyway'